In [7]:
# ============================================================
# CELL 01 — ENVIRONMENT
# ============================================================

!pip install -U scikit-learn

import os
import random
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)


print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Scikit-learn:", __import__("sklearn").__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ENVIRONMENT
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
Scikit-learn: 1.9.0
CUDA available: False


In [8]:
# ====================================================
# CEL 02 - CONFIGURATION + REPRODUCIBILITY
# ====================================================

SEED = 42

# Dataset
DATA_DIR = Path("/content/drive/MyDrive/face_frames")

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"
TEST_DIR = DATA_DIR / "test"

# Image
IMG_SIZE = 224

#Training
BATCH_SIZE = 32
NUM_EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4

# DataLoader
NUM_WORKERS = 2

# Classification
CLASS_NAMES = ["REAL", "FAKE"]
NUM_CLASSES = 1

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)

  torch.manual_seed(seed)

  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)

print("DATA_DIR:", DATA_DIR)
print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("LR:", LR)
print("NUM_WORKERS:", NUM_WORKERS)
print("DEVICE:", device)

DATA_DIR: /content/drive/MyDrive/face_frames
IMG_SIZE: 224
BATCH_SIZE: 32
NUM_EPOCHS: 10
LR: 0.001
NUM_WORKERS: 2
DEVICE: cpu


In [9]:
# ============================================================
# CELL 03 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ============================================================
# CELL 04 — VERIFY DATASET
# ============================================================

for name, path in {
    "TRAIN": TRAIN_DIR,
    "VAL": VAL_DIR,
    "TEST": TEST_DIR,
}.items():

    print(
        f"{name:>5}: "
        f"{'OK' if path.exists() else 'MISSING'}"
        f"  -> {path}"
    )

    if path.exists():
        print("       Classes:", sorted(
            [p.name for p in path.iterdir() if p.is_dir()]
        ))

TRAIN: OK  -> /content/drive/MyDrive/face_frames/train
       Classes: ['FAKE', 'REAL']
  VAL: OK  -> /content/drive/MyDrive/face_frames/val
       Classes: ['FAKE', 'REAL']
 TEST: OK  -> /content/drive/MyDrive/face_frames/test
       Classes: ['FAKE', 'REAL']


In [ ]:
# ============================================================
# CELL 05 — DATASET STATISTICS
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def get_dataset_statistics(split_dir):
    rows = []

    for class_name in CLASS_NAMES:

        class_dir = split_dir / class_name

        if not class_dir.exists():
            continue

        video_count = 0
        frame_count = 0

        for video_dir in class_dir.iterdir():

            if not video_dir.is_dir():
                continue

            video_count += 1

            for image_path in video_dir.iterdir():

                if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                    frame_count += 1

        rows.append({
            "Class": class_name,
            "Videos": video_count,
            "Frames": frame_count,
        })

    return pd.DataFrame(rows)


for split_name, split_dir in [
    ("TRAIN", TRAIN_DIR),
    ("VAL", VAL_DIR),
    ("TEST", TEST_DIR),
]:

    print(f"\n{'=' * 20} {split_name} {'=' * 20}")

    df = get_dataset_statistics(split_dir)

    display(df)


==================== TRAIN ====================


In [ ]:
# ============================================================
# CELL 06 — DEEPFAKE DATASET
# ============================================================

class DeepfakeDataset(Dataset):

    LABEL_MAP = {
        "REAL": 0,
        "FAKE": 1,
    }

    def __init__(self, root_dir, transform=None):

        self.root_dir = Path(root_dir)
        self.transform = transform

        self.samples = []

        for class_name, label in self.LABEL_MAP.items():

            class_dir = self.root_dir / class_name

            if not class_dir.exists():
                continue

            for video_dir in sorted(class_dir.iterdir()):

                if not video_dir.is_dir():
                    continue

                for image_path in sorted(video_dir.iterdir()):

                    if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                        continue

                    self.samples.append({
                        "image_path": image_path,
                        "label": label,
                        "video_id": video_dir.name,
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        sample = self.samples[index]

        with Image.open(sample["image_path"]) as img:
            image = img.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,
            "label": torch.tensor(
                sample["label"],
                dtype=torch.float32
            ),
            "video_id": sample["video_id"],
            "image_path": str(sample["image_path"]),
        }

In [ ]:
# ============================================================
# CELL 07 — TRANSFORMS
# ============================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])


eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD,
    ),
])

In [ ]:
# ============================================================
# CELL 08 — CREATE DATASETS
# ============================================================

train_dataset = DeepfakeDataset(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = DeepfakeDataset(
    VAL_DIR,
    transform=eval_transform
)

test_dataset = DeepfakeDataset(
    TEST_DIR,
    transform=eval_transform
)

print("Train:", len(train_dataset))
print("Val:  ", len(val_dataset))
print("Test: ", len(test_dataset))

In [ ]:
# ============================================================
# CELL 09 — CLASS DISTRIBUTION
# ============================================================

train_labels = [
    sample["label"]
    for sample in train_dataset.samples
]

class_counts = np.bincount(train_labels)

for label, count in enumerate(class_counts):

    print(
        f"{CLASS_NAMES[label]:>5}: "
        f"{count:,} "
        f"({count / len(train_labels):.2%})"
    )

In [ ]:
# ============================================================
# CELL 09B — WEIGHTED RANDOM SAMPLER
# ============================================================

class_weights = 1.0 / class_counts

sample_weights = torch.tensor(
    [class_weights[label] for label in train_labels],
    dtype=torch.double
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

print("WeightedRandomSampler: READY")

In [ ]:
# ============================================================
# CELL 10 — DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))
print("Test batches: ", len(test_loader))

In [ ]:
# ============================================================
# CELL 11 — SANITY CHECK
# ============================================================

batch = next(iter(train_loader))

print("Image shape :", batch["image"].shape)
print("Label shape :", batch["label"].shape)
print("Labels      :", batch["label"][:10])
print("Video IDs   :", batch["video_id"][:5])

In [ ]:
# ============================================================
# CELL 12 — DATALOADER BENCHMARK
# ============================================================

def benchmark_loader(loader, num_batches=50):

    start = time.time()

    for i, batch in enumerate(loader):

        if i + 1 >= num_batches:
            break

    elapsed = time.time() - start

    print(f"Batches tested : {min(num_batches, len(loader))}")
    print(f"Total time     : {elapsed:.2f} s")
    print(
        f"Average/batch  : "
        f"{elapsed / min(num_batches, len(loader)):.4f} s"
    )


benchmark_loader(train_loader, num_batches=50)

In [ ]:
# ============================================================
# CELL 13 — MODEL FACTORY
# ============================================================

def create_model(model_name):

    if model_name == "resnet50":

        model = models.resnet50(
            weights=models.ResNet50_Weights.DEFAULT
        )

        in_features = model.fc.in_features

        model.fc = nn.Linear(
            in_features,
            1
        )

    elif model_name == "efficientnet_b0":

        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )

        in_features = model.classifier[1].in_features

        model.classifier[1] = nn.Linear(
            in_features,
            1
        )

    elif model_name == "convnext_tiny":

        model = models.convnext_tiny(
            weights=models.ConvNeXt_Tiny_Weights.DEFAULT
        )

        in_features = model.classifier[2].in_features

        model.classifier[2] = nn.Linear(
            in_features,
            1
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return model

In [ ]:
# ============================================================
# CELL 14 — FREEZE BACKBONE
# ============================================================

def freeze_backbone(model):

    for param in model.parameters():
        param.requires_grad = False

    if hasattr(model, "fc"):

        for param in model.fc.parameters():
            param.requires_grad = True

    elif hasattr(model, "classifier"):

        for param in model.classifier.parameters():
            param.requires_grad = True

    else:
        raise ValueError(
            "Cannot identify classification head."
        )

    return model

In [ ]:
# ============================================================
# CELL 15 — PARAMETER CHECK
# ============================================================

def inspect_model(model_name):

    model = create_model(model_name)
    model = freeze_backbone(model)

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    ratio = (
        100.0 * trainable_params / total_params
    )

    print(f"\nModel: {model_name}")
    print(f"Total parameters     : {total_params:,}")
    print(f"Trainable parameters : {trainable_params:,}")
    print(f"Trainable ratio      : {ratio:.4f}%")


for name in [
    "resnet50",
    "efficientnet_b0",
    "convnext_tiny",
]:

    inspect_model(name)

In [ ]:
# ============================================================
# CELL 16 — TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
):

    model.train()

    running_loss = 0.0
    num_samples = 0

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(images).squeeze(1)

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        optimizer.step()

        batch_size = images.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        num_samples += batch_size

    return running_loss / num_samples

In [ ]:
# ============================================================
# CELL 17 — EVALUATION
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
):

    model.eval()

    running_loss = 0.0
    num_samples = 0

    all_labels = []
    all_probs = []

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].to(
            device,
            non_blocking=True
        )

        logits = model(images).squeeze(1)

        loss = criterion(
            logits,
            labels
        )

        batch_size = images.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        num_samples += batch_size

        probs = torch.sigmoid(logits)

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_probs.extend(
            probs.cpu().numpy()
        )

    labels = np.asarray(all_labels)
    probs = np.asarray(all_probs)

    predictions = (
        probs >= 0.5
    ).astype(int)

    return {
        "loss": running_loss / num_samples,

        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            labels,
            predictions,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            labels,
            probs
        ),
    }

In [ ]:
# ============================================================
# CELL 18 — TRAIN MODEL
# ============================================================

def train_model(
    model_name,
    num_epochs=NUM_EPOCHS,
):

    print("\n" + "=" * 70)
    print(f"TRAINING: {model_name}")
    print("=" * 70)

    model = create_model(model_name)
    model = freeze_backbone(model)
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    trainable_params = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_f1 = -1.0
    best_epoch = 0
    best_state = None

    history = []

    total_start = time.time()

    for epoch in range(num_epochs):

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
        )

        val_metrics = evaluate(
            model,
            val_loader,
            criterion,
        )

        epoch_time = (
            time.time() - epoch_start
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "epoch_time_sec": epoch_time,
        })

        print(
            f"Epoch [{epoch+1:02d}/{num_epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"F1: {val_metrics['f1']:.4f} | "
            f"AUC: {val_metrics['roc_auc']:.4f} | "
            f"Time: {epoch_time:.1f}s"
        )

        if val_metrics["f1"] > best_f1:

            best_f1 = val_metrics["f1"]
            best_epoch = epoch + 1

            best_state = copy.deepcopy(
                model.state_dict()
            )

    total_time = time.time() - total_start

    model.load_state_dict(best_state)

    history = pd.DataFrame(history)

    print(
        f"\nBest epoch: {best_epoch}"
        f" | Best Val F1: {best_f1:.4f}"
    )

    print(
        f"Total training time: "
        f"{total_time / 60:.2f} min"
    )

    return model, history

In [ ]:
# ============================================================
# CELL 19 — PILOT TEST
# ============================================================

pilot_model, pilot_history = train_model(
    "resnet50",
    num_epochs=1,
)

In [ ]:
# ============================================================
# CELL 20 — BASELINE: RESNET50
# ============================================================

resnet50_model, resnet50_history = train_model(
    "resnet50"
)

In [ ]:
# ============================================================
# CELL 21 — BASELINE: EFFICIENTNET-B0
# ============================================================

efficientnet_model, efficientnet_history = train_model(
    "efficientnet_b0"
)

In [ ]:
# ============================================================
# CELL 22 — BASELINE: CONVNEXT-TINY
# ============================================================

convnext_model, convnext_history = train_model(
    "convnext_tiny"
)

In [ ]:
# ============================================================
# CELL 23 — VALIDATION COMPARISON
# ============================================================

def get_best_metrics(history):

    best_idx = history["val_f1"].idxmax()
    best_row = history.loc[best_idx]

    return {
        "Best Epoch": int(best_row["epoch"]),
        "Accuracy": best_row["val_accuracy"],
        "Precision": best_row["val_precision"],
        "Recall": best_row["val_recall"],
        "F1": best_row["val_f1"],
        "ROC-AUC": best_row["val_roc_auc"],
        "Epoch Time (s)": best_row["epoch_time_sec"],
    }


comparison = pd.DataFrame({
    "ResNet50": get_best_metrics(
        resnet50_history
    ),

    "EfficientNet-B0": get_best_metrics(
        efficientnet_history
    ),

    "ConvNeXt-Tiny": get_best_metrics(
        convnext_history
    ),
}).T

comparison = comparison.sort_values(
    "F1",
    ascending=False,
)

display(comparison)

In [ ]:
# ============================================================
# CELL 24 — SELECT BEST MODEL
# ============================================================

best_model_name = comparison.index[0]

print("=" * 60)
print("BEST BASELINE MODEL")
print("=" * 60)

print("Model:", best_model_name)

display(
    comparison.loc[[best_model_name]]
)

In [ ]:
# ============================================================
# CELL 25 — FINAL TEST
# ============================================================

model_dict = {
    "ResNet50": resnet50_model,
    "EfficientNet-B0": efficientnet_model,
    "ConvNeXt-Tiny": convnext_model,
}

best_model = model_dict[best_model_name]

criterion = nn.BCEWithLogitsLoss()

test_metrics = evaluate(
    best_model,
    test_loader,
    criterion,
)

print("=" * 60)
print("FINAL TEST RESULT")
print("=" * 60)

for metric, value in test_metrics.items():

    print(
        f"{metric:>10}: {value:.4f}"
    )

In [ ]:
# ============================================================
# CELL 26 — CONFUSION MATRIX
# ============================================================

@torch.no_grad()
def get_predictions(model, loader):

    model.eval()

    all_labels = []
    all_probs = []

    for batch in loader:

        images = batch["image"].to(
            device,
            non_blocking=True
        )

        labels = batch["label"].cpu().numpy()

        logits = model(images).squeeze(1)

        probs = torch.sigmoid(logits)

        all_labels.extend(labels)

        all_probs.extend(
            probs.cpu().numpy()
        )

    labels = np.asarray(all_labels)
    probs = np.asarray(all_probs)

    predictions = (
        probs >= 0.5
    ).astype(int)

    return labels, predictions, probs


labels, predictions, probs = get_predictions(
    best_model,
    test_loader,
)

cm = confusion_matrix(
    labels,
    predictions,
)

plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.xticks(
    [0, 1],
    CLASS_NAMES
)

plt.yticks(
    [0, 1],
    CLASS_NAMES
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title(
    f"Confusion Matrix - {best_model_name}"
)

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
        )

plt.colorbar()
plt.show()